In [18]:
# ── Block 1: Install Dependencies ────────────────────────────────────────
!pip install -q --upgrade qiskit qiskit-aer qiskit-ibm-runtime

In [19]:
# ── Block 2: Imports & Constants ─────────────────────────────────────────
import numpy as np
import random
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.circuit import Measure
from kaggle_secrets import UserSecretsClient

SHOTS     = 1024
TOLERANCE = 2.0   # 2.0 = default | 1.5 = stricter | 3.0 = looser

In [20]:
# ── Block 3: Connect to IBM Backend ──────────────────────────────────────
secrets = UserSecretsClient()
token   = secrets.get_secret("IBM_QUANTUM_TOKEN")

service = QiskitRuntimeService(token=token, instance="open-instance")
backend = service.least_busy(simulator=False, operational=True, min_num_qubits=2)
pm      = generate_preset_pass_manager(backend=backend, optimization_level=1,
                                       translation_method="translator")

print(f"Backend : {backend.name}")
print(f"Qubits  : {backend.num_qubits}")
print(f"Pending : {backend.status().pending_jobs}")

qiskit_runtime_service._discover_account:WARNING:2026-06-09 16:40:26,048: Loading account with the given token. A saved account will not be used.


Backend : ibm_fez
Qubits  : 156
Pending : 3


In [21]:
# ── Block 4: Define Your Circuit ─────────────────────────────────────────
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  ONLY EDIT THIS BLOCK. All blocks below run automatically on it.    ║
# ║  Rules:                                                             ║
# ║    1. Set NQUBITS to match your circuit width                       ║
# ║    2. Assign circuit_correct  → your reference / correct circuit    ║
# ║    3. Assign circuit_buggy    → your faulty / mutated circuit       ║
# ╚══════════════════════════════════════════════════════════════════════╝

NQUBITS = 2   # <── change this if your circuit uses more qubits

# --- Example: 2-qubit Grover ---
# Replace build_grover_2q() with any circuit builder you want.

def build_grover_2q(inject_x_on=None):
    qc = QuantumCircuit(NQUBITS, NQUBITS)
    qc.h([0, 1])
    qc.cz(0, 1)
    qc.h([0, 1])
    qc.x([0, 1])
    qc.h(1)
    qc.cx(0, 1)
    qc.h(1)
    qc.x([0, 1])
    qc.h([0, 1])
    if inject_x_on is not None:
        qc.x(inject_x_on)
    qc.measure(range(NQUBITS), range(NQUBITS))
    return qc

bug_qubit       = random.choice(range(NQUBITS))
circuit_correct = build_grover_2q()
circuit_buggy   = build_grover_2q(inject_x_on=bug_qubit)

print("Correct circuit:")
print(circuit_correct.draw(output="text", fold=-1))
print(f"\nBuggy circuit (X injected on q{bug_qubit}):")
print(circuit_buggy.draw(output="text", fold=-1))

Correct circuit:
     ┌───┐   ┌───┐┌───┐          ┌───┐┌───┐     ┌─┐   
q_0: ┤ H ├─■─┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────┤M├───
     ├───┤ │ ├───┤├───┤┌───┐┌─┴─┐├───┤├───┤┌───┐└╥┘┌─┐
q_1: ┤ H ├─■─┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├─╫─┤M├
     └───┘   └───┘└───┘└───┘└───┘└───┘└───┘└───┘ ║ └╥┘
c: 2/════════════════════════════════════════════╩══╩═
                                                 0  1 

Buggy circuit (X injected on q1):
     ┌───┐   ┌───┐┌───┐          ┌───┐┌───┐     ┌─┐        
q_0: ┤ H ├─■─┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────┤M├────────
     ├───┤ │ ├───┤├───┤┌───┐┌─┴─┐├───┤├───┤┌───┐└╥┘┌───┐┌─┐
q_1: ┤ H ├─■─┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├─╫─┤ X ├┤M├
     └───┘   └───┘└───┘└───┘└───┘└───┘└───┘└───┘ ║ └───┘└╥┘
c: 2/════════════════════════════════════════════╩═══════╩═
                                                 0       1 


In [22]:
# ── Block 5: Ideal Distribution (Noiseless Simulator) ────────────────────
# Reads: circuit_correct, NQUBITS, SHOTS
# Writes: probs_ideal

def counts_to_probs(counts, nqubits):
    total = sum(counts.values())
    probs = np.zeros(2 ** nqubits)
    for bitstring, count in counts.items():
        probs[int(bitstring, 2)] = count / total
    return probs

simulator_ideal = AerSimulator()
t_ideal         = transpile(circuit_correct, simulator_ideal)
counts_ideal    = simulator_ideal.run(t_ideal, shots=SHOTS).result().get_counts()
probs_ideal     = counts_to_probs(counts_ideal, NQUBITS)

print("Ideal distribution:")
for i, p in enumerate(probs_ideal):
    if p > 1e-4:
        print(f"  |{format(i, f'0{NQUBITS}b')}> {p:.4f}")

Ideal distribution:
  |11> 1.0000


In [23]:
# ── Block 6: Run Both Circuits on Real IBM Hardware ───────────────────────
# Reads: circuit_correct, circuit_buggy, NQUBITS, SHOTS, backend, pm
# Writes: probs_hw_correct, probs_hw_buggy

def sampler_to_counts(result_entry):
    try:
        return result_entry.data.c.get_counts()
    except AttributeError:
        pass
    try:
        reg = list(result_entry.data.keys())[0]
        return result_entry.data[reg].get_counts()
    except Exception:
        pass
    raise RuntimeError("Cannot extract counts from SamplerV2 result.")

t_correct_hw = pm.run(circuit_correct)
t_buggy_hw   = pm.run(circuit_buggy)

sampler = Sampler(mode=backend)
job     = sampler.run([t_correct_hw, t_buggy_hw], shots=SHOTS)
result  = job.result()

counts_hw_correct = sampler_to_counts(result[0])
counts_hw_buggy   = sampler_to_counts(result[1])
probs_hw_correct  = counts_to_probs(counts_hw_correct, NQUBITS)
probs_hw_buggy    = counts_to_probs(counts_hw_buggy,   NQUBITS)

print("Hardware distribution — correct circuit:")
for i, p in enumerate(probs_hw_correct):
    if p > 1e-4:
        print(f"  |{format(i, f'0{NQUBITS}b')}> {p:.4f}")

print("\nHardware distribution — buggy circuit:")
for i, p in enumerate(probs_hw_buggy):
    if p > 1e-4:
        print(f"  |{format(i, f'0{NQUBITS}b')}> {p:.4f}")

# ── IBM Quantum Usage Summary ─────────────────────────────────────────────
from datetime import datetime

try:
    metrics    = job.metrics()
    usage_secs = metrics.get("usage", {}).get("seconds", None)
    if usage_secs is None:
        t_start    = metrics.get("timestamps", {}).get("running",  None)
        t_end      = metrics.get("timestamps", {}).get("finished", None)
        if t_start and t_end:
            fmt        = "%Y-%m-%dT%H:%M:%S.%fZ"
            usage_secs = (datetime.strptime(t_end,   fmt) -
                          datetime.strptime(t_start, fmt)).total_seconds()
except Exception:
    usage_secs = None

try:
    u              = service.usage()
    quota_secs     = u.get("usage_limit_seconds",     None)
    consumed_secs  = u.get("usage_consumed_seconds",  None)
    remaining_secs = u.get("usage_remaining_seconds", None)
except Exception:
    quota_secs     = None
    consumed_secs  = None
    remaining_secs = None

print("\n" + "=" * 50)
print("       IBM Quantum — Usage Summary")
print("=" * 50)

if usage_secs is not None:
    m, s = divmod(int(usage_secs), 60)
    print(f"  This job used  : {usage_secs:.2f}s  ({m}m {s}s)")
else:
    print("  This job used  : unavailable")

if consumed_secs is not None:
    cm, cs = divmod(int(consumed_secs), 60)
    print(f"  Total consumed : {consumed_secs:.2f}s  ({cm}m {cs}s)")
else:
    print("  Total consumed : unavailable")

if quota_secs is not None:
    qm, qs = divmod(int(quota_secs), 60)
    print(f"  Monthly quota  : {quota_secs:.2f}s  ({qm}m {qs}s)")
else:
    print("  Monthly quota  : unavailable")

if remaining_secs is not None:
    rm, rs  = divmod(int(remaining_secs), 60)
    pct_str = f"  ({remaining_secs / quota_secs * 100:.1f}% left)" if quota_secs else ""
    print(f"  Remaining      : {remaining_secs:.2f}s  ({rm}m {rs}s){pct_str}")
else:
    print("  Remaining      : unavailable")

if usage_secs and remaining_secs:
    print(f"  Est. runs left : ~{int(remaining_secs // usage_secs)} more runs at this size")

print("=" * 50)

Hardware distribution — correct circuit:
  |00> 0.0049
  |01> 0.0625
  |10> 0.1328
  |11> 0.7998

Hardware distribution — buggy circuit:
  |00> 0.1289
  |01> 0.8213
  |10> 0.0088
  |11> 0.0410

       IBM Quantum — Usage Summary
  This job used  : 3.00s  (0m 3s)
  Total consumed : 50.00s  (0m 50s)
  Monthly quota  : 600.00s  (10m 0s)
  Remaining      : 550.00s  (9m 10s)  (91.7% left)
  Est. runs left : ~183 more runs at this size


In [24]:
# ── Block 7: Per-Qubit IBM Error Model (Self-Calibrating) ─────────────────
# Reads: circuit_correct, NQUBITS, backend, pm, probs_ideal, probs_hw_correct
# Writes: gate_err, readout_err, combined_err, used_qubits, virt_to_phys

# ── marginal_p0 defined here so Block 7 can use it for self-calibration ───
def marginal_p0(probs, nqubits, q):
    return sum(
        probs[i]
        for i in range(2 ** nqubits)
        if format(i, f"0{nqubits}b")[nqubits - 1 - q] == "0"
    )

def get_circuit_aware_error(tqc, backend, observed_flips=None):
    props = backend.properties()

    # ── SPAM: readout + state preparation error per qubit ─────────────────
    readout = {}
    for q, qprops in enumerate(props.qubits):
        ro = next((float(p.value) for p in qprops if p.name == "readout_error"), None)
        if ro is None:
            m01 = next((float(p.value) for p in qprops if p.name == "prob_meas0_prep1"), None)
            m10 = next((float(p.value) for p in qprops if p.name == "prob_meas1_prep0"), None)
            ro  = (m01 + m10) / 2.0 if (m01 and m10) else 0.0
        sp         = next((float(p.value) for p in qprops if p.name == "prob_meas1_prep0"), 0.0)
        readout[q] = 1.0 - (1.0 - ro) * (1.0 - sp)

    # ── Gate error + duration accumulation ────────────────────────────────
    log_fid   = {}
    gate_time = {}

    for inst in tqc.data:
        if isinstance(inst.operation, Measure):
            continue
        name    = inst.operation.name
        phys_qs = [tqc.find_bit(q).index for q in inst.qubits]

        try:
            err = float(props.gate_error(name, phys_qs))
        except Exception:
            try:
                err = float(props.gate_error("sx", [phys_qs[0]]))
            except Exception:
                err = 0.0

        try:
            duration = float(props.gate_length(name, phys_qs))
        except Exception:
            duration = 0.0

        for pq in phys_qs:
            if pq not in log_fid:
                log_fid[pq]   = 0.0
                gate_time[pq] = 0.0
            log_fid[pq]   += np.log(max(1.0 - err / len(phys_qs), 1e-12))
            gate_time[pq] += duration

    # ── T1/T2 decoherence over full circuit time (active + idle) ──────────
    total_circuit_time = max(gate_time.values()) if gate_time else 0.0

    for pq in list(log_fid.keys()):
        try:
            t1       = float(props.t1(pq))
            t2       = float(props.t2(pq))
            t2_eff   = min(t2, 2.0 * t1)
            decohere = 1.0 - np.exp(-total_circuit_time / t2_eff)
        except Exception:
            decohere = 0.0
        log_fid[pq] += np.log(max(1.0 - decohere, 1e-12))

    # ── Raw combined error ─────────────────────────────────────────────────
    used         = sorted(log_fid.keys())
    gate_err     = {q: 1.0 - np.exp(log_fid[q]) for q in used}
    combined_raw = {q: 1.0 - (1.0 - gate_err[q]) * (1.0 - readout[q]) for q in used}

    # ── Self-calibration: drift correction using correct circuit ──────────
    if observed_flips is not None:
        drift = {}
        for v, pq in enumerate(used):
            raw = combined_raw[pq]
            obs = observed_flips.get(v, raw)
            drift[pq] = np.clip(obs / raw, 0.5, 2.0) if raw > 1e-9 else 1.0
        combined = {q: np.clip(combined_raw[q] * drift[q], 0.0, 1.0) for q in used}
    else:
        drift    = {q: 1.0 for q in used}
        combined = combined_raw

    return gate_err, readout, combined, used, drift


# ── Step 1: first pass without calibration ────────────────────────────────
t_correct_tp                                         = pm.run(circuit_correct)
gate_err, readout_err, combined_raw, used_qubits, _  = get_circuit_aware_error(
    t_correct_tp, backend, observed_flips=None
)
virt_to_phys = {v: used_qubits[v] for v in range(NQUBITS)}

# ── Step 2: compute flip_correct for self-calibration ─────────────────────
flip_correct_calibration = {}
for q in range(NQUBITS):
    p0i                       = marginal_p0(probs_ideal,      NQUBITS, q)
    p0c                       = marginal_p0(probs_hw_correct, NQUBITS, q)
    flip_correct_calibration[q] = abs(p0c - p0i)

# ── Step 3: second pass WITH self-calibration ─────────────────────────────
gate_err, readout_err, combined_err, used_qubits, drift = get_circuit_aware_error(
    t_correct_tp, backend, observed_flips=flip_correct_calibration
)
virt_to_phys = {v: used_qubits[v] for v in range(NQUBITS)}

# ── Print results ─────────────────────────────────────────────────────────
print(f"Physical qubits used: {used_qubits}")
print(f"\n{'Qubit':>8}  {'Gate+Decohere':>14}  {'Readout+SP':>12}  {'Combined Err':>13}  {'Drift Factor':>13}")
print("-" * 70)
for v in range(NQUBITS):
    pq = virt_to_phys[v]
    print(f"  q{v}→q{pq:<3}  {gate_err[pq]:>14.6f}  {readout_err[pq]:>12.6f}  {combined_err[pq]:>13.6f}  {drift[pq]:>13.4f}")

Physical qubits used: [0, 1]

   Qubit   Gate+Decohere    Readout+SP   Combined Err   Drift Factor
----------------------------------------------------------------------
  q0→q0          0.024543      0.014008       0.076413         2.0000
  q1→q1          0.013908      0.031982       0.067383         1.4827


In [25]:
# ── Block 8: Per-Qubit Observed Flip Rate ────────────────────────────────
# Reads: probs_ideal, probs_hw_correct, probs_hw_buggy, NQUBITS
# Writes: flip_correct, flip_buggy
# Note: marginal_p0 is defined in Block 7

flip_correct = {}
flip_buggy   = {}
for q in range(NQUBITS):
    p0i              = marginal_p0(probs_ideal,      NQUBITS, q)
    flip_correct[q]  = abs(marginal_p0(probs_hw_correct, NQUBITS, q) - p0i)
    flip_buggy[q]    = abs(marginal_p0(probs_hw_buggy,   NQUBITS, q) - p0i)

print(f"\n{'Q':>4}  {'P0 ideal':>10}  {'P0 hw_cor':>10}  {'flip(cor)':>10}  {'P0 hw_bug':>10}  {'flip(bug)':>10}")
print("-" * 70)
for q in range(NQUBITS):
    p0i = marginal_p0(probs_ideal,      NQUBITS, q)
    p0c = marginal_p0(probs_hw_correct, NQUBITS, q)
    p0b = marginal_p0(probs_hw_buggy,   NQUBITS, q)
    print(f"  q{q}  {p0i:>10.4f}  {p0c:>10.4f}  {flip_correct[q]:>10.4f}  {p0b:>10.4f}  {flip_buggy[q]:>10.4f}")


   Q    P0 ideal   P0 hw_cor   flip(cor)   P0 hw_bug   flip(bug)
----------------------------------------------------------------------
  q0      0.0000      0.1377      0.1377      0.1377      0.1377
  q1      0.0000      0.0674      0.0674      0.9502      0.9502


In [26]:
# ── Block 9: IBM Combined Error vs Observed Flip Rate ────────────────────
# Reads: combined_err, virt_to_phys, flip_correct, NQUBITS

hw_mean   = np.mean([combined_err[virt_to_phys[q]] for q in range(NQUBITS)])
flip_mean = np.mean([flip_correct[q] for q in range(NQUBITS)])

RANK_TOLERANCE = 0.005  # gaps below this are treated as consistent

print("\nIBM Combined Error  vs  Observed Flip Rate (correct circuit)")
print("=" * 62)
print(f"  {'VQ':>2}  {'PQ':>3}  {'IBM Combined':>13}  {'Flip Rate':>10}  Consistent?")
print("-" * 62)
for v in range(NQUBITS):
    pq  = virt_to_phys[v]
    ce  = combined_err[pq]
    fr  = flip_correct[v]
    gap = abs(ce - fr)
    ok  = gap < RANK_TOLERANCE or (ce < hw_mean) == (fr < flip_mean)
    print(f"  q{v}  q{pq:<3}  {ce:>13.6f}  {fr:>10.6f}  {'yes' if ok else 'no'}")
print("=" * 62)

safest   = min(range(NQUBITS), key=lambda q: combined_err[virt_to_phys[q]])
riskiest = max(range(NQUBITS), key=lambda q: combined_err[virt_to_phys[q]])
print(f"\n  Safest   → q{safest}  combined={combined_err[virt_to_phys[safest]]:.6f}  flip={flip_correct[safest]:.6f}")
print(f"  Riskiest → q{riskiest}  combined={combined_err[virt_to_phys[riskiest]]:.6f}  flip={flip_correct[riskiest]:.6f}")


IBM Combined Error  vs  Observed Flip Rate (correct circuit)
  VQ   PQ   IBM Combined   Flip Rate  Consistent?
--------------------------------------------------------------
  q0  q0         0.076413    0.137695  yes
  q1  q1         0.067383    0.067383  yes

  Safest   → q1  combined=0.067383  flip=0.067383
  Riskiest → q0  combined=0.076413  flip=0.137695


In [27]:
# ── Block 10: Per-Qubit Bug Detection via Threshold ──────────────────────
# Reads: combined_err, virt_to_phys, flip_buggy, NQUBITS, TOLERANCE
#
# threshold[q] = combined_err[q] × TOLERANCE
# flip_buggy[q] > threshold[q]  →  BUG
# flip_buggy[q] <= threshold[q] →  hardware noise
#
# Tune TOLERANCE in Block 2:
#   2.0 = default  |  1.5 = stricter  |  3.0 = looser

print(f"\nPer-Qubit Bug Detection  (TOLERANCE = {TOLERANCE}x)")
print("=" * 68)
print(f"  {'Q':>2}  {'Combined Err':>13}  {'Threshold':>10}  {'flip(bug)':>10}  Verdict")
print("-" * 68)

any_bug = False
for q in range(NQUBITS):
    pq        = virt_to_phys[q]
    ce        = combined_err[pq]
    threshold = ce * TOLERANCE
    fb        = flip_buggy[q]
    bug       = fb > threshold
    if bug:
        any_bug = True
    print(f"  q{q}  {ce:>13.6f}  {threshold:>10.6f}  {fb:>10.6f}  {'BUG' if bug else 'noise'}")

print("=" * 68)
print(f"\n  threshold[q] = combined_err[q] × {TOLERANCE}")
print(f"  OVERALL VERDICT: {'BUG DETECTED' if any_bug else 'NO BUG — within hardware noise'}")


Per-Qubit Bug Detection  (TOLERANCE = 2.0x)
   Q   Combined Err   Threshold   flip(bug)  Verdict
--------------------------------------------------------------------
  q0       0.076413    0.152826    0.137695  noise
  q1       0.067383    0.134766    0.950195  BUG

  threshold[q] = combined_err[q] × 2.0
  OVERALL VERDICT: BUG DETECTED
